# Notebook 04 — Recurrent Networks for Sequences

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 4 of 12  
**Time estimate:** 75 minutes

> RNNs process sequences step by step, maintaining a hidden state that summarizes
> everything seen so far. The LSTM gates solve the vanishing gradient problem
> that makes vanilla RNNs fail on long sequences — critical for protein sequences
> where dependencies can span hundreds of residues.

---
## Step 1 — Motivation

A convolutional layer has a fixed receptive field — it cannot capture long-range
dependencies beyond its kernel size without many stacked layers. Protein secondary
structure depends on residues 50–100 positions apart (helix packing, strand pairing).
A recurrent network maintains state across the full sequence, making it naturally
suited to long-range dependencies — until the vanishing gradient problem appears.
LSTMs solve this with gated memory.

---
## Step 2 — Intuition

**Vanilla RNN:** at each step $t$, update hidden state:
$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$
The hidden state summarizes the history. But gradients flow back through all $T$ steps:
if the largest singular value of $W_h$ is $< 1$, gradients vanish; if $> 1$, they explode.

**LSTM:** adds a *cell state* $C_t$ — a "conveyor belt" that carries information
across long distances with minimal transformation. Three gates control information flow:
- **Forget gate** $f_t$: what fraction of old memory to keep
- **Input gate** $i_t$: what new information to add to memory
- **Output gate** $o_t$: what part of memory to expose as hidden state

**GRU (Gated Recurrent Unit):** simplified LSTM — two gates, no separate cell state.
Slightly fewer parameters; often comparable performance.

**Bidirectional RNN:** run one RNN left-to-right, another right-to-left.
Concatenate hidden states — each position now sees context from both directions.
Essential for protein secondary structure (residue $i$ depends on residues both before and after).

---
## Step 3 — Biological Background

**Protein secondary structure prediction:**
Predict H (helix), E (strand), or C (coil) for each residue from the amino acid sequence.
- NetSurf-2.0, PSIPRED: use bidirectional LSTMs
- ProtTrans (Elnaggar 2021): protein language model (LSTM-based) pre-trained on UniProt
  → fine-tuned for secondary structure, achieves ~85% Q3 accuracy
- Local context window ≈ 15 residues helps a lot; 50+ residues add marginal gain
  (transformer-based models capture this better)

**Splicing prediction:**
Deep splicing (Leung 2014) used LSTMs on sequences around exon-intron boundaries.
The LSTM captures the AG-GT rule (2 residues) AND longer upstream context (30+ nt).

**Sequence annotation as sequence-to-sequence:**
Input: protein sequence (AA embedding at each position).
Output: secondary structure label per residue.
This is `many-to-many` RNN: an output at every timestep.
Contrast with TF binding (many-to-one: one output for the whole sequence).

---
## Step 4 — Mathematical Explanation

**LSTM equations (at time step $t$):**

$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(input gate)}$$
$$\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C) \quad \text{(candidate cell state)}$$
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad \text{(cell state update)}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(output gate)}$$
$$h_t = o_t \odot \tanh(C_t) \quad \text{(hidden state)}$$

Where $\odot$ is elementwise multiplication. $[h_{t-1}, x_t]$ is concatenation.

**Why LSTMs avoid vanishing gradients:**
The cell state update $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ is additive.
The gradient $\partial C_t / \partial C_{t-1} = f_t$ (the forget gate values).
If $f_t \approx 1$ ("remember everything"), gradients flow back with minimal attenuation.

In [ ]:
# Step 6 — Python: Bidirectional LSTM for protein secondary structure
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Amino acid vocabulary ----
AA = 'ACDEFGHIKLMNPQRSTVWY'  # 20 standard amino acids
AA_IDX = {a: i for i, a in enumerate(AA)}
SS_LABELS = {'H': 0, 'E': 1, 'C': 2}  # helix, strand, coil

# ---- Synthetic protein dataset ----
# Rules based on simplified secondary structure tendencies:
# Helix: AELM residues tend to form helices (simplified)
# Strand: IVT residues tend to form strands
# Coil: elsewhere

rng = np.random.default_rng(42)
HELIX_AA = set('AELM')
STRAND_AA = set('IVT')

def generate_protein_sequence(length=60):
    # Generate segments of H, E, C with minimum lengths
    seq, labels = [], []
    while len(seq) < length:
        ss = rng.choice(['H','E','C'], p=[0.35, 0.25, 0.40])
        seg_len = int(rng.integers(3, 12))
        if ss == 'H':
            residues = [rng.choice(list(HELIX_AA)) if rng.random() < 0.6
                         else rng.choice(list(AA)) for _ in range(seg_len)]
        elif ss == 'E':
            residues = [rng.choice(list(STRAND_AA)) if rng.random() < 0.6
                         else rng.choice(list(AA)) for _ in range(seg_len)]
        else:
            residues = rng.choice(list(AA), seg_len).tolist()
        seq.extend(residues[:length-len(seq)])
        labels.extend([ss]*len(residues[:length-len(seq)]))
    return ''.join(seq[:length]), labels[:length]

n_proteins = 500
SEQ_LEN = 60
proteins = [generate_protein_sequence(SEQ_LEN) for _ in range(n_proteins)]
print(f'Generated {n_proteins} synthetic proteins of length {SEQ_LEN}')
print(f'Example: {proteins[0][0]}')
print(f'Labels:  {"".join(proteins[0][1])}')

# ---- One-hot encode amino acids ----
def encode_protein(seq):
    """Encode as (L, 20) one-hot array."""
    X = np.zeros((len(seq), len(AA)), dtype=np.float32)
    for i, a in enumerate(seq):
        if a in AA_IDX:
            X[i, AA_IDX[a]] = 1.0
    return X

class ProteinDataset(Dataset):
    def __init__(self, proteins):
        self.X = [torch.tensor(encode_protein(seq), dtype=torch.float32) for seq, _ in proteins]
        self.y = [torch.tensor([SS_LABELS[s] for s in labels], dtype=torch.long) for _, labels in proteins]
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

dataset = ProteinDataset(proteins)
train_ds, val_ds = torch.utils.data.random_split(dataset, [400, 100])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

# ---- Bidirectional LSTM model ----
class BiLSTMSS(nn.Module):
    def __init__(self, input_size=20, hidden_size=64, n_layers=2, n_classes=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers,
                              batch_first=True, bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size * 2, n_classes)  # *2 for bidirectional
    
    def forward(self, x):
        out, _ = self.lstm(x)  # (batch, L, 2*hidden)
        return self.fc(out)    # (batch, L, n_classes)

model_lstm = BiLSTMSS().to(device)
print(f'BiLSTM parameters: {sum(p.numel() for p in model_lstm.parameters()):,}')

# ---- Training ----
optimizer_lstm = optim.Adam(model_lstm.parameters(), lr=1e-3)
criterion_lstm = nn.CrossEntropyLoss()

train_losses, val_accs = [], []
for epoch in range(30):
    model_lstm.train()
    bl = []
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_lstm.zero_grad()
        logits = model_lstm(X_b)  # (batch, L, 3)
        loss = criterion_lstm(logits.reshape(-1, 3), y_b.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model_lstm.parameters(), 1.0)  # gradient clipping
        optimizer_lstm.step()
        bl.append(loss.item())
    train_losses.append(np.mean(bl))
    
    model_lstm.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            preds = model_lstm(X_b).argmax(-1)  # (batch, L)
            correct += (preds == y_b).sum().item()
            total += y_b.numel()
    val_accs.append(correct / total)
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={train_losses[-1]:.4f}, val Q3={val_accs[-1]:.3f}')

print(f'\nFinal Q3 accuracy: {max(val_accs):.3f}')
print(f'Random baseline Q3: {1/3:.3f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
SS_COLORS = {0: 'tomato', 1: 'steelblue', 2: 'lightgrey'}
SS_NAMES = {0: 'Helix', 1: 'Strand', 2: 'Coil'}

# Panel A: Training curves
ax = axes[0]
ax.plot(train_losses, 'steelblue', lw=1.5, label='Train loss')
ax2 = ax.twinx()
ax2.plot(val_accs, 'tomato', lw=1.5, label='Val Q3 acc')
ax.set_xlabel('Epoch'); ax.set_ylabel('CE loss', color='steelblue')
ax2.set_ylabel('Q3 accuracy', color='tomato')
ax.set_title('A. BiLSTM training curves\n(secondary structure prediction)')
lines = [plt.Line2D([0],[0],color='steelblue',lw=2,label='Train loss'),
           plt.Line2D([0],[0],color='tomato',lw=2,label='Val Q3')]
ax.legend(handles=lines, fontsize=8, loc='lower left')

# Panel B: Prediction on one protein (sequence-to-sequence)
ax = axes[1]
model_lstm.eval()
seq0, labels0 = proteins[0]
X0 = torch.tensor(encode_protein(seq0), dtype=torch.float32).unsqueeze(0).to(device)
with torch.no_grad():
    preds0 = model_lstm(X0).argmax(-1).squeeze().cpu().numpy()
true0 = [SS_LABELS[s] for s in labels0]

for pos in range(SEQ_LEN):
    ax.bar(pos, 0.5, bottom=1, color=SS_COLORS[true0[pos]], edgecolor='none', width=1)
    ax.bar(pos, 0.5, bottom=0, color=SS_COLORS[preds0[pos]], edgecolor='none', width=1)
ax.set_xlim(0, SEQ_LEN)
ax.set_yticks([0.25, 1.25])
ax.set_yticklabels(['Predicted', 'True'], fontsize=9)
ax.set_xlabel('Sequence position')
ax.set_title('B. Predicted vs. true SS\n(per-residue for one protein)')
from matplotlib.patches import Patch
ax.legend([Patch(color=SS_COLORS[i]) for i in range(3)],
            [SS_NAMES[i] for i in range(3)], fontsize=8, loc='upper right')

# Panel C: Hidden state UMAP/PCA (residues colored by true SS)
from sklearn.decomposition import PCA
model_lstm.eval()
hidden_states = []
ss_labels_all = []
with torch.no_grad():
    for X_b, y_b in val_loader:
        out, _ = model_lstm.lstm(X_b.to(device))  # (batch, L, 2*hidden)
        hidden_states.append(out.cpu().numpy().reshape(-1, 128))
        ss_labels_all.extend(y_b.numpy().ravel().tolist())
hidden_all = np.vstack(hidden_states)
ss_labels_all = np.array(ss_labels_all)

pca_h = PCA(n_components=2)
H_2d = pca_h.fit_transform(hidden_all[:2000])  # subset for speed
ss_2d = ss_labels_all[:2000]
ax = axes[2]
for ss_i, ss_name in SS_NAMES.items():
    mask = ss_2d == ss_i
    ax.scatter(H_2d[mask, 0], H_2d[mask, 1], c=SS_COLORS[ss_i], s=5, alpha=0.5, label=ss_name)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('C. BiLSTM hidden states (PCA)\n(colored by true secondary structure)')
ax.legend(fontsize=9)

plt.suptitle('Module 14 NB04: Recurrent Networks for Sequences', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('recurrent_networks.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement the LSTM update equations from scratch in numpy for a single timestep.
   Verify the cell state and hidden state match `torch.nn.LSTMCell` on the same inputs.
2. Compare vanilla RNN vs. LSTM on this dataset. On which sequence lengths does
   the vanilla RNN fail? Use gradient norm tracking to show the vanishing gradient.
3. Add a window of 5 neighboring residues as context (concat surrounding amino acids).
   Does this improve Q3 accuracy on the validation set?
4. Replace the LSTM with a GRU (`nn.GRU`). Compare parameter count, training speed,
   and final Q3 accuracy. Which is more efficient here?

---
## Step 10 — Quiz

1. Write the six LSTM equations. What is the role of each gate?
2. Why does the vanilla RNN suffer from vanishing gradients? What is the mathematical
   condition under which gradients vanish?
3. Why does the LSTM cell state update avoid the vanishing gradient problem?
4. What is a bidirectional RNN? Why is it important for sequence annotation tasks?
5. What is gradient clipping and why is it used with RNNs?

---
## Step 12 — Reflection

> *[In one paragraph: explain the LSTM cell state $C_t$ as a biological analogy —
> compare it to epigenetic memory, where past signals leave marks that influence
> future gene expression without changing the sequence.]*

---
*Next: `05_attention_and_transformers.ipynb`*